# FluxPipeline - Batch Image Generation

This notebook demonstrates how to efficiently generate multiple images in batch.

## What You'll Learn
- Generate multiple images with different prompts
- Batch processing best practices
- Memory management for batch operations
- Organize and save multiple outputs

## Setup

In [ ]:
# Add parent directory to path
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import torch
import matplotlib.pyplot as plt
from datetime import datetime
import gc

from pipeline import FluxPipeline
from core import SeedProfile
from config import setup_environment, logger
from utils import setup_workspace

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Initialize Pipeline

In [ ]:
setup_environment()
workspace = setup_workspace()

# Create a batch-specific output directory
batch_dir = workspace / f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
batch_dir.mkdir(exist_ok=True)
print(f"Batch output directory: {batch_dir}")

pipeline = FluxPipeline(workspace=workspace)
pipeline.load_model()
print("✅ Pipeline ready!")

## Method 1: Simple Batch Generation

Generate multiple images with a list of prompts.

In [ ]:
# Define your batch of prompts
prompts = [
    "A serene mountain landscape at sunrise",
    "A futuristic cityscape with flying cars",
    "An enchanted forest with glowing mushrooms",
    "A steampunk airship floating in clouds",
    "An underwater coral reef teeming with life",
]

print(f"Generating {len(prompts)} images...")

results = []

for idx, prompt in enumerate(prompts, 1):
    print(f"\n[{idx}/{len(prompts)}] Generating: {prompt[:50]}...")
    
    image, seed = pipeline.generate_image(
        prompt=prompt,
        num_inference_steps=4,
        height=768,
        width=768,
        seed_profile=SeedProfile.BALANCED
    )
    
    if image:
        # Save with descriptive filename
        filename = f"batch_{idx:02d}_seed_{seed}.png"
        output_path = batch_dir / filename
        image.save(output_path)
        
        results.append({
            'image': image,
            'prompt': prompt,
            'seed': seed,
            'path': output_path
        })
        print(f"  ✅ Saved: {filename}")
    else:
        print(f"  ❌ Failed")

print(f"\n✅ Generated {len(results)}/{len(prompts)} images successfully!")

## Display Results

In [ ]:
# Display all generated images
num_images = len(results)
cols = 3
rows = (num_images + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axes = axes.flatten() if num_images > 1 else [axes]

for idx, result in enumerate(results):
    axes[idx].imshow(result['image'])
    axes[idx].axis('off')
    axes[idx].set_title(
        f"{result['prompt'][:40]}...\nSeed: {result['seed']}",
        fontsize=9
    )

# Hide unused subplots
for idx in range(num_images, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## Method 2: Batch with Variations

Generate multiple variations of the same prompt using different seeds.

In [ ]:
# Single prompt, multiple variations
base_prompt = "A magical crystal cave with bioluminescent plants"
num_variations = 4

print(f"Generating {num_variations} variations of:\n'{base_prompt}'\n")

variations = []

for i in range(num_variations):
    print(f"Variation {i+1}/{num_variations}...")
    
    image, seed = pipeline.generate_image(
        prompt=base_prompt,
        num_inference_steps=4,
        height=768,
        width=768,
        seed_profile=SeedProfile.CREATIVE  # Use CREATIVE for more variation
    )
    
    if image:
        variations.append({'image': image, 'seed': seed})
        print(f"  ✅ Generated with seed {seed}")

print(f"\n✅ Generated {len(variations)} variations!")

In [ ]:
# Display variations side by side
fig, axes = plt.subplots(1, num_variations, figsize=(20, 5))

for idx, var in enumerate(variations):
    axes[idx].imshow(var['image'])
    axes[idx].axis('off')
    axes[idx].set_title(f"Seed: {var['seed']}", fontsize=10)

plt.suptitle(base_prompt, fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Method 3: Batch with Different Parameters

Test different generation parameters to find optimal settings.

In [ ]:
test_prompt = "A majestic dragon perched on a mountain peak"

# Test different resolutions
resolutions = [
    (512, 512),
    (768, 768),
    (1024, 1024),
]

# Use same seed for fair comparison
fixed_seed = 12345

resolution_tests = []

for height, width in resolutions:
    print(f"Generating {width}x{height}...")
    
    image, seed = pipeline.generate_image(
        prompt=test_prompt,
        seed=fixed_seed,
        height=height,
        width=width,
        num_inference_steps=4
    )
    
    if image:
        resolution_tests.append({
            'image': image,
            'resolution': f"{width}x{height}"
        })
        print(f"  ✅ Done")

# Display resolution comparison
fig, axes = plt.subplots(1, len(resolution_tests), figsize=(18, 6))

for idx, test in enumerate(resolution_tests):
    axes[idx].imshow(test['image'])
    axes[idx].axis('off')
    axes[idx].set_title(f"{test['resolution']}", fontsize=12)

plt.suptitle(f"Resolution Comparison (seed: {fixed_seed})", fontsize=14)
plt.tight_layout()
plt.show()

## Memory Management Tips

When generating many images, manage GPU memory carefully.

In [ ]:
def generate_batch_with_cleanup(prompts, **kwargs):
    """Generate batch with automatic memory cleanup."""
    results = []
    
    for idx, prompt in enumerate(prompts, 1):
        print(f"[{idx}/{len(prompts)}] {prompt[:50]}...")
        
        image, seed = pipeline.generate_image(prompt=prompt, **kwargs)
        
        if image:
            # Save immediately to reduce memory usage
            filename = batch_dir / f"batch_cleanup_{idx:02d}_{seed}.png"
            image.save(filename)
            results.append({'seed': seed, 'path': filename})
            print(f"  ✅ Saved: {filename.name}")
            
        # Clear memory after each generation
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    
    return results

# Test with large batch
large_batch_prompts = [
    "A cyberpunk street market at night",
    "An ancient temple in a jungle",
    "A space station orbiting a ringed planet",
]

batch_results = generate_batch_with_cleanup(
    large_batch_prompts,
    height=768,
    width=768,
    num_inference_steps=4
)

print(f"\n✅ Batch complete! Generated {len(batch_results)} images")

## Export Batch Metadata

Save generation parameters for reproducibility.

In [ ]:
import json

# Create metadata file
metadata = {
    'batch_id': datetime.now().isoformat(),
    'total_images': len(results),
    'images': [
        {
            'filename': result['path'].name,
            'prompt': result['prompt'],
            'seed': result['seed']
        }
        for result in results
    ]
}

metadata_path = batch_dir / 'metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Metadata saved to: {metadata_path}")
print(f"\nBatch summary:")
print(f"  - Directory: {batch_dir}")
print(f"  - Images: {len(results)}")
print(f"  - Metadata: {metadata_path.name}")

## Next Steps

- **[03_gif_creation.ipynb](03_gif_creation.ipynb)** - Create animated GIFs from batches
- **[04_custom_prompts.ipynb](04_custom_prompts.ipynb)** - Advanced prompt engineering
- **[05_memory_optimization.ipynb](05_memory_optimization.ipynb)** - Optimize for large batches

## Cleanup

In [ ]:
# Clean up GPU memory
del pipeline
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ Cleanup complete!")